# Rveda Training Smoke Launcher

This notebook is a thin launcher for the Task 3.3 smoke run.

It installs the runtime dependencies, checks that the repo is visible, and then calls `train_grpo_smoke.py`.
It does not duplicate the training logic.


## 1. Install runtime dependencies

Run this once per Colab session.


In [ ]:
from pathlib import Path
import subprocess
import sys

packages = [
    "torch",
    "datasets",
    "accelerate",
    "trl",
    "unsloth",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Installed:", ", ".join(packages))

## 2. Confirm the repo is visible

The notebook expects `train_grpo_smoke.py` to be present in the current workspace or mounted folder.


In [ ]:
repo_root = Path.cwd()
script_path = repo_root / "train_grpo_smoke.py"
if not script_path.exists():
    candidates = [
        Path("/content/rveda"),
        Path("/content/drive/MyDrive/rveda"),
        Path("/workspace/rveda"),
    ]
    for candidate in candidates:
        if (candidate / "train_grpo_smoke.py").exists():
            repo_root = candidate
            script_path = candidate / "train_grpo_smoke.py"
            break

if not script_path.exists():
    raise FileNotFoundError("Could not find train_grpo_smoke.py. Clone or mount the repo before running this notebook.")

%cd {repo_root}
print("Repo root:", repo_root)
print("Script path:", script_path)

## 3. Launch the smoke runner

This calls the existing script with a minimal Colab-friendly configuration.


In [ ]:
output_dir = repo_root / "artifacts" / "grpo_smoke_colab"
command = [
    sys.executable,
    str(script_path),
    "--model-name",
    "Qwen/Qwen2.5-7B-Instruct",
    "--output-dir",
    str(output_dir),
    "--task-ids",
    "v2_easy_overweight_schema_v1",
    "--samples-per-task",
    "1",
    "--episodes",
    "1",
    "--train-steps",
    "1",
    "--max-episode-steps",
    "2",
]
print("Running:", " ".join(command))
subprocess.check_call(command)

## 4. Inspect the generated artifacts


In [ ]:
summary_path = output_dir / "summary.json"
if not summary_path.exists():
    raise FileNotFoundError(f"Missing summary artifact: {summary_path}")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))
print("Artifacts:")
for file_name in ["scripted_baseline.json", "baseline_model_eval.json", "post_train_model_eval.json", "summary.json"]:
    print("-", output_dir / file_name)